# CareLoom: Long-Term Customer Memory — Larkspur Electronics

A support thread ends when the ticket closes, but the customer doesn't start over from
zero the next time they write in. If Asha Rao told CareLoom last month that she prefers
a replacement over a refund for defective items, that preference should still be there
the next time she opens a ticket — even in a brand-new conversation thread. This notebook
shows a practical long-term memory setup with LangGraph:

- `SqliteSaver` for checkpointed thread state
- a persistent Chroma-based semantic memory index
- resume from checkpoint
- memory summarisation

LangGraph's persistence docs say checkpointers provide short-term, thread-scoped memory while stores provide long-term, cross-thread memory. The checkpointer docs also show `SqliteSaver` as one of the provided checkpoint libraries, and the memory docs describe adding long-term memory, semantic search, and summarizing messages. The time-travel docs show how checkpoint history can be inspected and how a prior checkpoint can be resumed.

## Learning goals

By the end of this notebook, you should be able to:

1. Persist thread state with `SqliteSaver`.
2. Store long-term memories in a semantic index.
3. Recall memories across turns and across threads.
4. Summarize conversation history into a compact memory.
5. Inspect checkpoint history and resume from a prior checkpoint.

## 1) Install packages


In [ ]:
%pip install -qU langgraph langgraph-checkpoint-sqlite langchain langchain-core langchain-chroma chromadb langchain-huggingface sentence-transformers python-dotenv numpy


## 2) Environment variables

A Groq key is optional in this notebook. The memory mechanics work locally even without a model key — the "answers" below are template strings that show what was retrieved, not LLM output.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))
print("LANGSMITH_API_KEY set:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT", "careloom-long-term-memory"))


## 3) Local persistence setup


In [ ]:
import sqlite3
import shutil
from pathlib import Path
from datetime import datetime, timezone
from typing_extensions import TypedDict

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

CHECKPOINT_DB = DATA_DIR / "memory_checkpoints.db"
AUDIT_DB = DATA_DIR / "memory_audit.db"
MEMORY_DIR = Path("./chroma_memory_store")

if CHECKPOINT_DB.exists():
    CHECKPOINT_DB.unlink()
if AUDIT_DB.exists():
    AUDIT_DB.unlink()
if MEMORY_DIR.exists():
    shutil.rmtree(MEMORY_DIR)

print("Checkpoint DB:", CHECKPOINT_DB.resolve())
print("Audit DB:", AUDIT_DB.resolve())
print("Memory dir:", MEMORY_DIR.resolve())


## 4) Semantic memory store

We use a persistent Chroma collection as a local semantic memory index — the same offline-safe embeddings pattern used in the earlier notebooks: try the real `sentence-transformers` model, fall back to a dependency-free hashing embedding if it can't be reached.


In [ ]:
import hashlib
import numpy as np
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma


class LocalHashEmbeddings(Embeddings):
    """
    Deterministic, dependency-free fallback embedding function. Used only when
    the real sentence-transformers model can't be reached. Mechanically sound
    for demonstrating similarity search, but semantically weaker than a real
    sentence embedding.
    """

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str) -> list[float]:
        vec = np.zeros(self.dim)
        for token in text.lower().split():
            h = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16)
            vec[h % self.dim] += 1.0
        norm = np.linalg.norm(vec)
        return (vec / norm).tolist() if norm > 0 else vec.tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


def build_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        emb.embed_query("connectivity check")
        print("Using live HuggingFace embeddings (sentence-transformers/all-MiniLM-L6-v2).")
        return emb
    except Exception as e:
        print(f"[offline-fallback] HuggingFace embeddings unavailable ({type(e).__name__}: {e}).")
        print("[offline-fallback] Using LocalHashEmbeddings instead.")
        return LocalHashEmbeddings()


embeddings = build_embeddings()

memory_store = Chroma(
    collection_name="long_term_memory",
    persist_directory=str(MEMORY_DIR),
    embedding_function=embeddings,
)

print("Memory store ready")


## 5) Tiny audit table

This makes it easy to show what the graph saves over time.


In [ ]:
conn = sqlite3.connect(AUDIT_DB)
cur = conn.cursor()

cur.execute(
    "CREATE TABLE memory_audit ("
    "id INTEGER PRIMARY KEY AUTOINCREMENT, "
    "user_id TEXT NOT NULL, "
    "thread_id TEXT NOT NULL, "
    "memory_text TEXT NOT NULL, "
    "created_at TEXT NOT NULL"
    ")"
)

conn.commit()
conn.close()

print("Audit DB created")


## 6) Graph state


In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str
    summary: str
    memory_context: str
    response: str


## 7) Helper functions

The persistence docs distinguish short-term state in the checkpoint from durable memories in the store. The store docs also show semantic search over embedded memories.


In [ ]:
def latest_user_message(state: MemoryState) -> str:
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            return msg.content
        if isinstance(msg, dict) and msg.get("role") == "user":
            return msg.get("content", "")
    return ""

def lookup_memories(user_id: str, query: str, k: int = 3) -> str:
    results = memory_store.similarity_search(
        query=query,
        k=k,
        filter={"user_id": user_id},
    )
    if not results:
        return ""
    return "\n".join(f"- {doc.page_content}" for doc in results)

def summarize_history(messages: list, previous_summary: str = "") -> str:
    recent = []
    for msg in messages[-6:]:
        if isinstance(msg, HumanMessage):
            recent.append(f"user: {msg.content}")
        elif isinstance(msg, AIMessage):
            recent.append(f"assistant: {msg.content}")
        elif isinstance(msg, dict):
            role = msg.get("role", "unknown")
            recent.append(f"{role}: {msg.get('content', '')}")

    recent_text = "\n".join(recent)

    if previous_summary:
        return (previous_summary + " | " + recent_text)[:500]
    return recent_text[:500]

def store_memory(user_id: str, thread_id: str, memory_text: str, kind: str = "summary") -> None:
    doc = Document(
        page_content=memory_text,
        metadata={
            "user_id": user_id,
            "thread_id": thread_id,
            "kind": kind,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    memory_store.add_documents([doc])

    conn = sqlite3.connect(AUDIT_DB)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO memory_audit (user_id, thread_id, memory_text, created_at) VALUES (?, ?, ?, ?)",
        (user_id, thread_id, memory_text, datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()
    conn.close()


## 8) Graph nodes

We keep the graph simple:

1. retrieve relevant memories
2. answer the customer
3. summarize the conversation
4. store the summary as long-term memory

The memory docs describe summarizing messages after enough conversation has accumulated, using the previous summary as context for the next one.


In [ ]:
from langchain_core.runnables import RunnableConfig

def retrieve_memory(state: MemoryState):
    query = latest_user_message(state)
    context = lookup_memories(state["user_id"], query, k=3)
    return {"memory_context": context}

def answer_user(state: MemoryState):
    question = latest_user_message(state)
    summary = state.get("summary", "")
    memory_context = state.get("memory_context", "")

    response = (
        f"Summary so far: {summary[:140]}\n"
        f"Relevant memories: {memory_context[:160]}\n"
        f"Answer to: {question}"
    )
    return {
        "response": response,
        "messages": [AIMessage(content=response)],
    }

def summarize_conversation(state: MemoryState):
    new_summary = summarize_history(state["messages"], state.get("summary", ""))
    return {"summary": new_summary}

def save_long_term_memory(state: MemoryState, config: RunnableConfig):
    user_id = state["user_id"]
    thread_id = config["configurable"].get("thread_id", "unknown-thread")
    memory_text = state.get("summary", "") or latest_user_message(state)
    store_memory(user_id, thread_id, memory_text, kind="summary")
    return {}


## 9) Build the graph with `SqliteSaver`

The checkpointer docs say a thread ID is required and that `graph.get_state_history(config)` gives the checkpoint history for that thread. They also say a prior `checkpoint_id` can be used for replay.


In [ ]:
builder = StateGraph(MemoryState)

builder.add_node("retrieve_memory", retrieve_memory)
builder.add_node("answer_user", answer_user)
builder.add_node("summarize_conversation", summarize_conversation)
builder.add_node("save_long_term_memory", save_long_term_memory)

builder.add_edge(START, "retrieve_memory")
builder.add_edge("retrieve_memory", "answer_user")
builder.add_edge("answer_user", "summarize_conversation")
builder.add_edge("summarize_conversation", "save_long_term_memory")
builder.add_edge("save_long_term_memory", END)

checkpointer = SqliteSaver(sqlite3.connect(str(CHECKPOINT_DB), check_same_thread=False))
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled with SqliteSaver.")


## 10) First conversation turn

Asha Rao — a returning customer from Chennai (the same customer from the database-connected agent notebook earlier in this module) — mentions a standing preference during a support conversation.


In [ ]:
config = {"configurable": {"thread_id": "support-thread-1"}}

turn1 = graph.invoke(
    {
        "messages": [HumanMessage(content="Hi, I'm Asha from Chennai -- I usually prefer a replacement over a refund when something's defective.")],
        "user_id": "asha-rao",
        "summary": "",
        "memory_context": "",
        "response": "",
    },
    config=config,
)

turn1


## 11) Second turn

Still the same thread, so the checkpointed `summary` and `memory_context` carry forward.


In [ ]:
turn2 = graph.invoke(
    {
        "messages": [HumanMessage(content="I'm reaching out again about a damaged keyboard -- what would you do?")],
        "user_id": "asha-rao",
        "summary": turn1.get("summary", ""),
        "memory_context": turn1.get("memory_context", ""),
        "response": "",
    },
    config=config,
)

turn2


## 12) View checkpoint history

The newest checkpoint appears first in the list.


In [ ]:
history = list(graph.get_state_history(config))
print("Checkpoint count:", len(history))

for i, snap in enumerate(history[:3], 1):
    print(f"--- Snapshot {i} ---")
    print("checkpoint_id:", snap.config["configurable"]["checkpoint_id"])
    print("next:", snap.next)
    print("summary:", snap.values.get("summary", "")[:120])
    print()


## 13) Resume from a prior checkpoint

The time-travel docs describe resuming from a selected checkpoint and exploring an alternate path. The checkpointer docs also explain that replay re-executes nodes after the chosen checkpoint.


In [ ]:
if len(history) >= 2:
    old_checkpoint_id = history[-2].config["configurable"]["checkpoint_id"]

    replay_config = {
        "configurable": {
            "thread_id": "support-thread-1",
            "checkpoint_id": old_checkpoint_id,
        }
    }

    replayed = graph.invoke(
        {
            "messages": [HumanMessage(content="If I ask again, what do you remember about me?")],
            "user_id": "asha-rao",
            "summary": history[-2].values.get("summary", ""),
            "memory_context": history[-2].values.get("memory_context", ""),
            "response": "",
        },
        config=replay_config,
    )

    replayed
else:
    print("Not enough checkpoints yet to replay.")


## 14) Inspect the semantic memory store


In [ ]:
direct_hits = memory_store.similarity_search(
    "What does the customer prefer for defective items?",
    k=3,
    filter={"user_id": "asha-rao"},
)

for i, doc in enumerate(direct_hits, 1):
    print(f"--- Memory hit {i} ---")
    print(doc.page_content)
    print(doc.metadata)
    print()


## 15) Memory summarisation


In [ ]:
print("Current summary:")
print(turn2.get("summary", ""))

print("\nAudit rows:")
conn = sqlite3.connect(AUDIT_DB)
cur = conn.cursor()
cur.execute("SELECT user_id, thread_id, memory_text FROM memory_audit")
for row in cur.fetchall():
    print(row)
conn.close()


## Key takeaways

- Checkpointers keep thread-scoped short-term state.
- Stores keep durable long-term memory across threads.
- Semantic search lets you recall memories by meaning — here, a customer's stated preference from one ticket, resurfaced in a completely different one.
- Summarization keeps old conversation context compact.
- `graph.get_state_history(config)` and a prior `checkpoint_id` let you inspect and resume a conversation.

### What's verified where

| Piece | Status |
|---|---|
| Everything in this notebook | Executed and verified in this sandbox. No Groq call is involved anywhere — the "answers" are template strings built from retrieved memory and summary state, exactly as designed — so this behaves the same on any machine. |
| Chroma semantic memory search | Verified using the `LocalHashEmbeddings` fallback in this sandbox. On a machine with HuggingFace access, swap in the real `sentence-transformers` model for meaningfully better recall on paraphrased questions. |

## References

- Persistence: https://docs.langchain.com/oss/python/langgraph/persistence
- Checkpointers: https://docs.langchain.com/oss/python/langgraph/checkpointers
- Add memory: https://docs.langchain.com/oss/python/langgraph/add-memory
- Stores: https://docs.langchain.com/oss/python/langgraph/stores
- Time travel: https://docs.langchain.com/oss/python/langchain/frontend/time-travel
